# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [2]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU disponível: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 0
else:
    print('⚠️  GPU não encontrada — usando CPU (treinamento mais lento)')
    DEVICE = 'cpu'


✅ GPU disponível: NVIDIA GeForce RTX 3070
   VRAM: 8.6 GB


## 3. Baixar Datasets

In [2]:
import kagglehub, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from roboflow import Roboflow

paths = {}

print("🔄 LAPAROSCOPIA Roboflow...")
rf = Roboflow(api_key="qYZBO5fqNIjOlDK6jH4o")
project = rf.workspace("laparoscopic-yolo").project("laparoscopy")
dataset = project.version(14).download("yolov8")
paths['cirurgia'] = './laparoscopy-14'
print(f'✅ cirurgia → {paths["cirurgia"]}[web:67]')

print("\n🔄 Kaggle outros...")
DATASETS_KAGGLE = {
    'consulta': 'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem': 'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download(item):
    context, dataset_id = item
    try:
        path = kagglehub.dataset_download(dataset_id)
        return context, path, None
    except Exception as e:
        return context, None, str(e)

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(download, item): item[0] for item in DATASETS_KAGGLE.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f'❌ {context}: {error}')
        else:
            paths[context] = path
            print(f'✅ {context} → {path}')

print(f'\n✅ TOTAL: {len(paths)}/4 datasets')
print("cirurgia: Roboflow Laparoscopia ✓")


🔄 LAPAROSCOPIA Roboflow...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Laparoscopy-14 in yolov8:: 100%|██████████| 2350/2350 [00:01<00:00, 1994.33it/s]


✅ cirurgia → ./laparoscopy-14[web:67]

🔄 Kaggle outros...
✅ fisioterapia → C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
✅ triagem → C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
✅ consulta → C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1

✅ TOTAL: 4/4 datasets
cirurgia: Roboflow Laparoscopia ✓


## 4. Inspecionar Estrutura dos Datasets

In [3]:
import os

print("📊 INSPEÇÃO DATASETS")
for context, path in paths.items():
    print(f'\n📁 {context.upper()} — {path}')
    if not os.path.exists(path):
        print("❌ Pasta não existe!")
        continue
        
    total_imgs, total_txts = 0, 0
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > 2:  # Limita profundidade
            continue
            
        indent = '  ' * level
        folder_name = os.path.basename(root)
        if folder_name:  # Evita root vazio
            print(f'{indent}{folder_name}/')
            
        # Conta IMGS e LABELS por pasta
        imgs = [f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg', '.jpg'))]
        txts = [f for f in files if f.endswith('.txt')]
        
        if imgs:
            print(f'{indent}  🖼️  {len(imgs)} imagens')
            total_imgs += len(imgs)
        if txts:
            print(f'{indent}  📝 {len(txts)} labels')
            total_txts += len(txts)
    
    # RESUMO por dataset
    print(f'{context.upper()}: 🖼️{total_imgs} imgs | 📝{total_txts} labels')
    if context == 'cirurgia':
        yaml_path = os.path.join(path, 'data.yaml')
        if os.path.exists(yaml_path):
            print(f"✅ data.yaml encontrado → YOLOv8 pronto!")
        else:
            print("⚠️  Sem data.yaml → Crie manualmente")

print("\n🎯 PRONTO PARA TREINO!")


📊 INSPEÇÃO DATASETS

📁 CIRURGIA — ./laparoscopy-14
laparoscopy-14/
  📝 2 labels
  test/
    images/
      🖼️  88 imagens
    labels/
      📝 88 labels
  train/
    images/
      🖼️  948 imagens
    labels/
      📝 948 labels
  valid/
    images/
      🖼️  133 imagens
    labels/
      📝 133 labels
CIRURGIA: 🖼️1169 imgs | 📝1171 labels
✅ data.yaml encontrado → YOLOv8 pronto!

📁 FISIOTERAPIA — C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
1/
  Blurred/
    Arm Raise Correct/
    Arm Raise Incorrect/
    Knee Extension Correct/
    Knee Extension Incorrect/
    Sit To Stand Correct/
    Sit To Stand Incorrect/
    train/
    val/
FISIOTERAPIA: 🖼️0 imgs | 📝0 labels

📁 TRIAGEM — C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
1/
  Aggressive_Poses_Dataset/
    images/
      🖼️  103 imagens
    labels/
      📝 103 labels
TRIAGEM: 🖼️103 imgs | 📝103 labels

📁 CONSULTA — C:\Use

In [4]:
import os, shutil, yaml
from pathlib import Path
import cv2

os.makedirs('runs/processed', exist_ok=True)

def analyze_dataset(path):
    """Análise completa com tipos YOLO"""
    p = Path(path)
    if not p.exists():
        return {'error': 'Pasta não existe'}
    
    stats = {
        'videos': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.mp4','.mov','.avi'}),
        'images': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.jpg','.jpeg','.png'}),
        'labels_txt': sum(1 for x in p.rglob('*.txt')),
        'yaml': bool(list(p.rglob('data.yaml'))),
        'train_split': bool(p / 'train'),
        'valid_split': bool(p / 'val'),
        'subdirs': [d.name for d in p.iterdir() if d.is_dir()],
        'classes_yaml': [],
        'samples': [f.name for f in sorted(p.rglob('*'))[:5]]
    }
    
    # Classes do YAML (se existir)
    yaml_file = next(p.rglob('data.yaml'), None)
    if yaml_file:
        with open(yaml_file) as f:
            data = yaml.safe_load(f)
            stats['classes_yaml'] = data.get('names', [])
    
    # Ratio imgs/labels (YOLO precisa 1:1)
    img_dirs = ['train/images', 'val/images', 'test/images']
    label_dirs = ['train/labels', 'val/labels', 'test/labels']
    stats['imgs_labels_ratio'] = 'OK' if stats['images'] == stats['labels_txt'] else f"{stats['images']}/{stats['labels_txt']}"
    
    return stats

# ========================================================
# ANALISA TODOS DATASETS
# ========================================================
print("🔍 ANÁLISE DATASETS YOLO")
print("="*60)
for context, path in paths.items():
    print(f"\n📁 {context.upper()}: {path}")
    stats = analyze_dataset(path)
    
    if 'error' in stats:
        print(f"  ❌ {stats['error']}")
        continue
    
    # RESUMO RÁPIDO
    print(f"  🖼️  Imagens: {stats['images']:,}")
    print(f"  📝 Labels: {stats['labels_txt']:,} (ratio: {stats['imgs_labels_ratio']})")
    print(f"  🎥 Vídeos: {stats['videos']}")
    print(f"  📄 YAML: {'✅' if stats['yaml'] else '❌'}")
    print(f"  📁 Splits: {'✅' if stats['train_split'] else '❌'} train | {'✅' if stats['valid_split'] else '❌'} val")
    
    if stats['classes_yaml']:
        print(f"  🏷️  Classes: {stats['classes_yaml']}")
    
    print(f"  📂 Pastas: {stats['subdirs'][:3]}{'...' if len(stats['subdirs'])>3 else ''}")

print("\n🎯 STATUS YOLOv8:")
for context in paths:
    if context == 'cirurgia':
        print(f"  CIRURGIA: {'🚀 PRONTO' if analyze_dataset(paths[context]).get('yaml') else '⚠️ Precisa YAML'}")


🔍 ANÁLISE DATASETS YOLO

📁 CIRURGIA: ./laparoscopy-14
  🖼️  Imagens: 1,169
  📝 Labels: 1,171 (ratio: 1169/1171)
  🎥 Vídeos: 0
  📄 YAML: ✅
  📁 Splits: ✅ train | ✅ val
  🏷️  Classes: ['Allis', 'Bag', 'Cautery', 'Forceps', 'Gallbladder', 'Liver', 'Suction']
  📂 Pastas: ['test', 'train', 'valid']

📁 FISIOTERAPIA: C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
  🖼️  Imagens: 0
  📝 Labels: 0 (ratio: OK)
  🎥 Vídeos: 339
  📄 YAML: ❌
  📁 Splits: ✅ train | ✅ val
  📂 Pastas: ['Blurred']

📁 TRIAGEM: C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
  🖼️  Imagens: 103
  📝 Labels: 103 (ratio: OK)
  🎥 Vídeos: 0
  📄 YAML: ✅
  📁 Splits: ✅ train | ✅ val
  🏷️  Classes: {0: 'person'}
  📂 Pastas: ['Aggressive_Poses_Dataset']

📁 CONSULTA: C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1
  🖼️  Imagens: 48,398
  📝 Labels: 96,796 (ratio: 48398/96796)
  🎥 Vídeos: 0
  📄 Y

## 5. Gerar data.yaml para Cada Contexto

In [6]:
import os, cv2, yaml, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

DATASET_OUT = Path("datasets/yolov8")
DATASET_OUT.mkdir(parents=True, exist_ok=True)

def safe_mkdir(p):
    """Cria diretório com parents=True seguro"""
    p.mkdir(parents=True, exist_ok=True)

def extract_frames(video_path, out_dir, step=5):
    cap = cv2.VideoCapture(str(video_path))
    frame_id = saved = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_id % step == 0:
            out = out_dir / f"{video_path.stem}_{frame_id:06d}.jpg"
            cv2.imwrite(str(out), frame)
            saved += 1
        frame_id += 1
    cap.release()
    return saved

def split_list(items, train_ratio=0.8):
    random.shuffle(items)
    n = int(len(items) * train_ratio)
    return items[:n], items[n:]

# =============================================================================
# PROCESSADORES (com error handling)
# =============================================================================
def process_cirurgia(path):
    print("🔧 CIRURGIA: Laparoscopia Roboflow")
    src = Path(path)
    out = DATASET_OUT / "cirurgia"
    safe_mkdir(out)
    shutil.copytree(src, out, dirs_exist_ok=True)
    
    yaml_path = out / "data.yaml"
    if yaml_path.exists():
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
        print(f"✅ Classes: {data.get('names', 'N/A')}")
    return "cirurgia", str(yaml_path)

def process_triagem(path):
    print("🔧 TRIAGEM: Aggressive poses")
    src = Path(path) / "Aggressive_Poses_Dataset"
    out = DATASET_OUT / "triagem"
    safe_mkdir(out)
    
    # Copia raw
    raw_out = out / "raw"
    safe_mkdir(raw_out)
    shutil.copytree(src, raw_out, dirs_exist_ok=True)
    
    # YAML
    classes = sorted(set(int(float(l.split()[0])) for lbl in raw_out.rglob("*.txt") 
                        for l in open(lbl) if l.strip()))
    data_yaml = {
        "path": str(out),
        "train": "raw/images/train",
        "val": "raw/images/val",
        "nc": len(classes),
        "names": {i: f"person_pose_{i}" for i in classes}
    }
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)
    return "triagem", str(yaml_path)

def process_fisioterapia(path):
    print("🔧 FISIOTERAPIA: Vídeos → frames")
    src = Path(path)
    out = DATASET_OUT / "fisioterapia"
    safe_mkdir(out)
    
    # Encontra vídeos
    videos = []
    for ext in ['*.mp4', '*.avi', '*.mov']:
        videos.extend(src.rglob(ext))
    print(f"🎥 {len(videos)} vídeos")
    
    if not videos:
        print("⚠️ Sem vídeos encontrados")
        return "fisioterapia", str(out)
    
    # Extrai frames
    temp_dir = out / "temp_frames"
    safe_mkdir(temp_dir)
    
    total_frames = 0
    with ThreadPoolExecutor(max_workers=2) as ex:  # Reduz workers
        futures = {ex.submit(extract_frames, v, temp_dir): v for v in videos[:20]}  # Limita 20 vídeos primeiro
        for f in futures:
            total_frames += f.result()
    
    print(f"🖼️ {total_frames} frames extraídos")
    
    # Organiza por classe (pastas)
    classes = [d for d in temp_dir.iterdir() if d.is_dir()]
    for cls in classes[:3]:  # Primeiras 3 classes
        frames = list(cls.glob("*.jpg"))
        if frames:
            train, val = split_list(frames)
            safe_mkdir(out / "train" / cls.name)
            safe_mkdir(out / "val" / cls.name)
            for f in train[:50]:  # Limita por classe
                shutil.move(str(f), out / "train" / cls.name)
            for f in val[:10]:
                shutil.move(str(f), out / "val" / cls.name)
    
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    # YAML classification
    class_names = [c.name for c in out.iterdir() if c.is_dir() and (c/"train").exists()]
    data_yaml = {"path": str(out), "train": "train", "val": "val", "nc": len(class_names), "names": class_names}
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)
    
    return "fisioterapia", str(yaml_path)

# EXECUTA
tasks = {
    "cirurgia": process_cirurgia,
    "triagem": process_triagem, 
    "fisioterapia": process_fisioterapia
}

yamls = {}
print("🚀 PREPROCESSAMENTO...")
for name, func in tasks.items():
    if name in paths:
        try:
            result = func(paths[name])
            yamls[result[0]] = result[1]
            print(f"✅ {name}")
        except Exception as e:
            print(f"❌ {name}: {e}")

print("\n📄 YAMLs:", yamls)


🚀 PREPROCESSAMENTO...
🔧 CIRURGIA: Laparoscopia Roboflow
✅ Classes: ['Allis', 'Bag', 'Cautery', 'Forceps', 'Gallbladder', 'Liver', 'Suction']
✅ cirurgia
🔧 TRIAGEM: Aggressive poses
✅ triagem
🔧 FISIOTERAPIA: Vídeos → frames
🎥 339 vídeos
🖼️ 578 frames extraídos
✅ fisioterapia

📄 YAMLs: {'cirurgia': 'datasets\\yolov8\\cirurgia\\data.yaml', 'triagem': 'datasets\\yolov8\\triagem\\data.yaml', 'fisioterapia': 'datasets\\yolov8\\fisioterapia\\data.yaml'}


## 6. Treinar os Modelos

In [ ]:
# ================================
# PIPELINE TREINO YOLOv8 - LAPAROSCOPIA FEMHEALTH
# ================================

from ultralytics import YOLO
import os, glob, traceback
from pathlib import Path

# CONFIG
DEVICE = 0  # GPU (muda para 'cpu' se erro)
BASE_DIR = Path("models/yolov8_femhealth")
BASE_DIR.mkdir(parents=True, exist_ok=True)

resultados = {}
status = {}

def safe_train(name, base_model, data_path, task="detect", epochs=30):
    """Treina YOLOv8 com skip inteligente + logs"""
    
    exp_dir = BASE_DIR / task / name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    # SKIP se best.pt existe
    best_pt = exp_dir / "weights" / "best.pt"
    if best_pt.exists():
        print(f"✅ {name.upper()} SKIP - {best_pt}")
        resultados[name] = str(exp_dir)
        return "SKIP"
    
    print(f"\n🚀 TREINO: {name.upper()} ({task})")
    print(f"📁 data: {data_path}")
    
    try:
        # Parâmetros otimizados
        params = {
            "data": data_path,
            "epochs": epochs,
            "imgsz": 640 if task != "classify" else 224,
            "batch": 16 if task == "detect" else 32,
            "device": DEVICE,
            "project": str(BASE_DIR),
            "name": f"{task}/{name}",
            "exist_ok": True,
            "patience": 8,
            "workers": 2,
            "save_period": 10,
            "task": task
        }
        
        model = YOLO(base_model)
        results = model.train(**params)
        
        resultados[name] = str(results.save_dir)
        print(f"✅ {name} salvo em {results.save_dir}")
        return "OK"
        
    except Exception as e:
        print(f"❌ {name}: {e}")
        traceback.print_exc()
        return "ERROR"

# ================================
# EXECUÇÃO SEQUENCIAL (sem crash)
# ================================
print("🔄 PIPELINE FEMHEALTH YOLOv8")

# 1. CIRURGIA LAPAROSCOPIA (PRIORIDADE)
if "cirurgia" in yamls:
    status["cirurgia"] = safe_train(
        name="laparoscopia", 
        base_model="yolov8n.pt",
        data_path=yamls["cirurgia"],
        task="detect",
        epochs=50  # Mais epochs pra instruments precisos
    )

# 2. TRIAGEM POSES AGRESSIVAS
if "triagem" in yamls:
    status["triagem"] = safe_train(
        name="aggressive_poses",
        base_model="yolov8n.pt",  # detect person
        data_path=yamls["triagem"],
        task="detect"
    )

# 3. FISIOTERAPIA (se processada)
if "fisioterapia" in yamls and Path(yamls["fisioterapia"]).exists():
    status["fisioterapia"] = safe_train(
        name="rehab_postures",
        base_model="yolov8n-cls.pt",
        data_path=yamls["fisioterapia"],
        task="classify",
        epochs=30
    )

print("\n" + "="*60)
print("🎯 RESUMO TREINOS")
print("="*60)
for name, st in status.items():
    path = resultados.get(name, "N/A")
    print(f"{name:<20} {st:<8} {path}")

print(f"\n🚀 Modelos em: {BASE_DIR}")
print("Use: YOLO('models/yolov8_femhealth/detect/laparoscopia/weights/best.pt')")


🔄 PIPELINE FEMHEALTH YOLOv8

🚀 TREINO: LAPAROSCOPIA (detect)
📁 data: datasets\yolov8\cirurgia\data.yaml
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\yolov8\cirurgia\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

## 7. Avaliar Métricas dos Modelos

In [ ]:
from ultralytics import YOLO

print(f'{'Contexto':<15} {'mAP50':>8} {'mAP50-95':>10} {'Precisão':>10} {'Recall':>8}')
print('-' * 55)

for context, model_path in resultados.items():
    model = YOLO(model_path)
    metrics = model.val(data=yamls[context], verbose=False)
    print(
        f'{context:<15}'
        f'{metrics.box.map50:>8.3f}'
        f'{metrics.box.map:>10.3f}'
        f'{metrics.box.mp:>10.3f}'
        f'{metrics.box.mr:>8.3f}'
    )


## 8. Testar com Imagem Real

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

os.makedirs('data/samples', exist_ok=True)

for context, model_path in resultados.items():
    # Busca primeira imagem disponível do dataset
    test_img = next(
        (os.path.join(root, f)
         for root, _, files in os.walk(paths[context])
         for f in files if f.endswith(('.jpg', '.png', '.jpeg'))),
        None
    )

    if not test_img:
        print(f'⚠️  Nenhuma imagem encontrada para {context}')
        continue

    model = YOLO(model_path)
    results = model(test_img, conf=0.25)
    out_path = f'data/samples/resultado_{context}.jpg'
    results[0].save(out_path)

    print(f'\n🔍 {context.upper()}')
    print(f'   Imagem: {os.path.basename(test_img)}')
    print(f'   Detecções: {len(results[0].boxes)}')
    display(Image(filename=out_path, width=400))


## 9. Resumo Final

In [ ]:
import os

print('\n📦 MODELOS TREINADOS:')
print('-' * 45)
for context in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    path = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {context:<15} {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {context:<15} não treinado')

print('\n🚀 Próximo passo: uvicorn app.main:app --reload')
